In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Load the data from the CSV file
df = pd.read_csv(
    "LUMEN_DS.csv",
    sep="|",
    quotechar='"',
    encoding="utf-16",
    low_memory=False,
)

In [ ]:
df.info()

In [ ]:
df["Product family"].unique()

In [ ]:
df["Product group"].unique()

In [ ]:
df["Item Code"].unique()

In [ ]:
df["CustomerID"].unique()

In [ ]:
minimum_frequency = 3

customer_frequency_table = (
    df["CustomerID"]
    .value_counts()
    .rename_axis("CustomerID")
    .reset_index(name="frequency")
    .sort_values(["frequency", "CustomerID"], ascending=[False, True])
    .reset_index(drop=True)
)

valid_customer_ids = customer_frequency_table.loc[
    customer_frequency_table["frequency"] >= minimum_frequency,
    "CustomerID",
]

original_row_count = len(df)
df_filtered = df[df["CustomerID"].isin(valid_customer_ids)].copy()
filtered_row_count = len(df_filtered)
removed_row_count = original_row_count - filtered_row_count
shrinkage_pct = (removed_row_count / original_row_count) * 100

print(f"CustomerID frequency table shape: {customer_frequency_table.shape}")
display(customer_frequency_table.head(20))
print(f"\nOriginal rows: {original_row_count:,}")
print(f"Rows after filtering CustomerID frequency >= {minimum_frequency}: {filtered_row_count:,}")
print(f"Removed rows: {removed_row_count:,}")
print(f"Dataset shrank by: {shrinkage_pct:.2f}%")
print(f"Unique CustomerID before: {df['CustomerID'].nunique():,}")
print(f"Unique CustomerID after: {df_filtered['CustomerID'].nunique():,}")


In [ ]:
invoice_frequency_table = (
    df["Invoice #"]
    .value_counts()
    .rename_axis("Invoice #")
    .reset_index(name="frequency")
    .sort_values(["frequency", "Invoice #"], ascending=[False, True])
    .reset_index(drop=True)
)

invoice_count = df["Invoice #"].nunique()

print(f"Number of unique invoices in the original dataset: {invoice_count:,}")
display(invoice_frequency_table.head(20))

plt.figure(figsize=(10, 6))
plt.hist(invoice_frequency_table["frequency"], bins=50, edgecolor="black")
plt.xlabel("Frequency of Invoice #")
plt.ylabel("Number of invoices")
plt.title("Histogram of Invoice # frequencies")
plt.show()


In [ ]:
from upselling_functions import build_product_after_last_purchase_matrix

product_after_last_matrix, product_index_lookup, customer_product_windows = build_product_after_last_purchase_matrix(
    df,
    min_customer_product_purchases=10,
    customer_col="CustomerID",
    product_col="Item Code",
    date_col="Invoice Date",
)

matrix_density = product_after_last_matrix.nnz / (
    product_after_last_matrix.shape[0] * product_after_last_matrix.shape[1]
)

print(f"Matrix shape: {product_after_last_matrix.shape}")
print(f"Non-zero entries: {product_after_last_matrix.nnz:,}")
print(f"Matrix density: {matrix_density:.6%}")
print(f"All products in matrix: {len(product_index_lookup):,}")
print(f"Products with at least one surviving customer-product pair: {customer_product_windows['Item Code'].nunique():,}")
print(f"Customer-product pairs kept: {len(customer_product_windows):,}")
print(f"Customers kept: {customer_product_windows['CustomerID'].nunique():,}")
display(product_index_lookup.head(20))
display(customer_product_windows.head(20))

matrix_coo = product_after_last_matrix.tocoo()

top_matrix_entries = pd.DataFrame(
    {
        "row_idx": matrix_coo.row,
        "col_idx": matrix_coo.col,
        "count": matrix_coo.data,
    }
)

top_matrix_entries = top_matrix_entries.sort_values(
    ["count", "row_idx", "col_idx"],
    ascending=[False, True, True],
).head(10)

lookup = product_index_lookup.rename(
    columns={"matrix_index": "matrix_idx", "Item Code": "item_code"}
)

top_matrix_entries = top_matrix_entries.merge(
    lookup,
    left_on="row_idx",
    right_on="matrix_idx",
    how="left",
).rename(columns={"item_code": "product_i"})

top_matrix_entries = top_matrix_entries.merge(
    lookup,
    left_on="col_idx",
    right_on="matrix_idx",
    how="left",
    suffixes=("", "_j"),
).rename(columns={"item_code": "product_j"})

print("Top 10 highest matrix elements:")
display(top_matrix_entries[["product_i", "product_j", "count"]])
